# Midclose Methodology — Refit & Reproducibility Check

Two questions:

1. **Option 1 — Refit predecessor's regression with the freshest training data?** Look at `Model/MIDCLOSE_PREDICTION_*.csv` (83 monthly rows FY19–FY25) and see if a regression on the per-date-component columns beats our calibration baseline.
2. **Option 2 — Can we reproduce the published WD-3/-2/-1 min/max table from raw weightage CSVs?** If our numbers differ, the published bounds are stale and recalibrating is a quick win.

## Headline finding

- **Option 1 is a dead end** — the "feature" columns in those CSVs are a *post-hoc decomposition* of the target, not predictive features. Demonstrated below.
- **Option 2 works** — the published min/max table is *partially* reproducible from FY23–FY25 raw counts. PROMISE_DATE values match closely; SERVICE/FIRD diverges materially. That divergence is the talking point.

In [1]:
import pandas as pd
import numpy as np
import re
import statsmodels.api as sm
from pathlib import Path

MIDCLOSE = Path('/Users/jasloop/Downloads/Midclose')
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

## Option 1 — Why the regression CSVs are a trap

`Model/MIDCLOSE_PREDICTION_Product.csv` has columns `MID_CLOSE_COUNT, PROMISE_DATE_COUNT, RETURN_HARDWARE_DATE_COUNT, FULFILLMENT_DATE_COUNT`. Regress the target on those features and you get R² ≈ 1.0 with every coefficient ≈ 1.0. That isn't a great model — it's because the columns *partition* the target.

In [2]:
prod = pd.read_csv(MIDCLOSE / 'Model/MIDCLOSE_PREDICTION_Product.csv').dropna()
svc  = pd.read_csv(MIDCLOSE / 'Model/MIDCLOSE_PREDICTIONS_Service.csv').dropna()

prod_feats = ['PROMISE_DATE_COUNT', 'RETURN_HARDWARE_DATE_COUNT', 'FULFILLMENT_DATE_COUNT']
svc_feats  = ['PROMISE_DATE_COUNT', 'FUTURE_INVOICE_DATE_COUNT', 'HOLD_RELEASE_DATE_COUNT', 'CONTRACT_START_DATE_COUNT', 'FULFILLMENT_DATE_COUNT']

prod_diff = prod['MID_CLOSE_COUNT'] - prod[prod_feats].sum(axis=1)
svc_diff  = svc['MID_CLOSE_COUNT']  - svc[svc_feats].sum(axis=1) - svc['UNKNOW_COUNT']

print('PRODUCT: MID_CLOSE_COUNT − sum(date_component_counts):')
print(prod_diff.describe().to_string())
print(f'\n   → residual ≈ 0 (max {prod_diff.abs().max():.0f} on totals up to {prod["MID_CLOSE_COUNT"].max():,.0f})')
print()
print('SERVICE: MID_CLOSE_COUNT − sum(date_component_counts) − UNKNOW_COUNT:')
print(svc_diff.describe().to_string())
print('\n   → identically zero. The "features" partition the target.')

PRODUCT: MID_CLOSE_COUNT − sum(date_component_counts):
count    69.000
mean      5.725
std      33.992
min       0.000
25%       0.000
50%       0.000
75%       2.000
max     282.000

   → residual ≈ 0 (max 282 on totals up to 32,986)

SERVICE: MID_CLOSE_COUNT − sum(date_component_counts) − UNKNOW_COUNT:
count   69.000
mean     0.000
std      0.000
min      0.000
25%      0.000
50%      0.000
75%      0.000
max      0.000

   → identically zero. The "features" partition the target.


In [3]:
X = sm.add_constant(prod[prod_feats].astype(float))
y = prod['MID_CLOSE_COUNT'].astype(float)
m = sm.OLS(y, X).fit()
print('PRODUCT regression — every coef ≈ 1.0, R² ≈ 1.0:')
print(f'  R² = {m.rsquared:.6f},  adj R² = {m.rsquared_adj:.6f}')
print(m.params.to_string())
print('\nWe gave it the answer split into pieces. To actually predict the future we need')
print('the WD-3/-2/-1 *eligible* counts, not post-close *actuals*. Those live in the Weightage CSVs.')

PRODUCT regression — every coef ≈ 1.0, R² ≈ 1.0:
  R² = 0.999961,  adj R² = 0.999959
const                        13.349
PROMISE_DATE_COUNT            1.000
RETURN_HARDWARE_DATE_COUNT    0.987
FULFILLMENT_DATE_COUNT        0.956

We gave it the answer split into pieces. To actually predict the future we need
the WD-3/-2/-1 *eligible* counts, not post-close *actuals*. Those live in the Weightage CSVs.


### Verdict on Option 1

The predecessor's per-day weights (0.42, 0.45, 0.65, ...) do **not** come from regressing `MID_CLOSE_COUNT` on the post-hoc decomposition. They come from a ratio in `Promise_Date_WD_All_Merged.csv` / `FIRD_WD_All_Merged.csv`:

$$\text{Weightage}_{\text{day}} = \frac{\text{Actuals\_Count}_{\text{day}}}{\text{Eligible\_Count}_{\text{day}}}$$

Each row is one calendar date × one fiscal period: *"on Aug 1 with promise-date in 23-Aug, 15 orders actually closed by EOM out of 28,471 eligible → ratio 0.00053."*

**That's the data with predictive content.** And it's the same kind of data our Java service starts gathering at 3pm Pacific now. So Option 1 reduces to: keep the cron running, build up an aligned `(eligible_count, actual_count)` dataset, then derive weights from it. No code change today.

## Option 2 — Reproduce the published min/max table

Per `Midclose Max Min Calculations.docx`:

$$\text{min}_{\text{WD}} = \frac{\sum \text{actuals before WD}}{\sum \text{actuals (entire period)}}, \quad \text{max}_{\text{WD}} = \frac{\sum \text{actuals from WD onward}}{\sum \text{actuals (entire period)}} + 1$$

Published table (FY23–FY25, 288,859 orders): WD-3 (0.24, 2.09), WD-2 (0.29, 1.65), WD-1 (0.44, 1.53).

In [4]:
PUBLISHED = {-3: (0.24, 2.09), -2: (0.29, 1.65), -1: (0.44, 1.53)}

def parse_wd(s):
    if pd.isna(s): return None
    m = re.match(r'WD([+-]?)(\d+)', str(s))
    if not m: return None
    sign = -1 if m.group(1) == '-' else 1
    return sign * int(m.group(2))

def compute_minmax(df, wd_col, count_col, label):
    df = df.copy()
    df['_wd'] = df[wd_col].apply(parse_wd)
    df['_cnt'] = pd.to_numeric(df[count_col], errors='coerce').fillna(0)
    df = df.dropna(subset=['_wd'])
    df['_wd'] = df['_wd'].astype(int)
    agg = df.groupby('_wd')['_cnt'].sum().sort_index()
    if agg.empty:
        print(f'\n=== {label} === (empty)')
        return
    total = agg.sum()
    print(f'\n=== {label} ===')
    print(f'  total actuals: {int(total):,}    WD range: {int(agg.index.min())}..{int(agg.index.max())}')
    print(f'  {"WD":>4} | {"actuals":>10} | {"min%":>7} | {"max%":>7}  | published')
    for wd in [-3, -2, -1, 0]:
        if wd not in agg.index: continue
        before = agg.loc[:wd-1].sum()
        from_here = agg.loc[wd:].sum()
        min_pct = before / total
        max_pct = (from_here / total) + 1
        pub = f'({PUBLISHED[wd][0]:.2f}, {PUBLISHED[wd][1]:.2f})' if wd in PUBLISHED else ''
        print(f'  {wd:>4} | {int(agg.loc[wd]):>10,} | {min_pct:>7.3f} | {max_pct:>7.3f}  | {pub}')

promise = pd.read_csv(MIDCLOSE / 'Weightage Calculations/Data Pre-Processing/Promise_Date_WD_All_Merged.csv')
fird    = pd.read_csv(MIDCLOSE / 'Weightage Calculations/Data Pre-Processing/FIRD_WD_All_Merged.csv')
flexi   = pd.read_csv(MIDCLOSE / 'Weightage Calculations/Data Pre-Processing/Flexi_WD.csv')

compute_minmax(promise, 'WORKDAY', 'Actuals_COUNT', 'PROMISE_DATE (PRODUCT side) FY23-25')
compute_minmax(fird,    'Workday', 'Actuals_Count', 'FIRD (SERVICE side)            FY23-25')
compute_minmax(flexi,   'WORKDAY', 'Count',         'FLEXI / BILL_RELEASE          FY23-25')


=== PROMISE_DATE (PRODUCT side) FY23-25 ===
  total actuals: 115,866    WD range: -30..3
    WD |    actuals |    min% |    max%  | published
    -3 |      5,316 |   0.300 |   1.700  | (0.24, 2.09)
    -2 |     11,403 |   0.346 |   1.654  | (0.29, 1.65)
    -1 |     14,341 |   0.444 |   1.556  | (0.44, 1.53)

=== FIRD (SERVICE side)            FY23-25 ===
  total actuals: 135,969    WD range: -30..3
    WD |    actuals |    min% |    max%  | published
    -3 |      2,499 |   0.177 |   1.823  | (0.24, 2.09)
    -2 |     10,992 |   0.195 |   1.805  | (0.29, 1.65)
    -1 |     58,988 |   0.276 |   1.724  | (0.44, 1.53)
     0 |     39,360 |   0.710 |   1.290  | 

=== FLEXI / BILL_RELEASE          FY23-25 ===
  total actuals: 11,032    WD range: -30..3
    WD |    actuals |    min% |    max%  | published
    -3 |         10 |   0.823 |   1.177  | (0.24, 2.09)
    -2 |          7 |   0.823 |   1.177  | (0.29, 1.65)
    -1 |        203 |   0.824 |   1.176  | (0.44, 1.53)
     0 |         59

## What the numbers say

| WD | Published | PROMISE only | FIRD only | Interpretation |
|---|---|---|---|---|
| -3 | (0.24, **2.09**) | (0.30, 1.70) | (0.18, **1.82**) | published is PRODUCT-biased |
| -2 | (0.29, **1.65**) | (0.35, 1.65) ✓ | (0.20, 1.81) | published max matches PROMISE exactly |
| -1 | (**0.44**, 1.53) | (0.44, 1.56) ✓ | (0.28, 1.72) | published min matches PROMISE exactly |

1. **Published table is anchored to PROMISE_DATE (PRODUCT side).** WD-2 max (1.65) and WD-1 min (0.44) match exactly. The predecessor shipped a PRODUCT-derived bound and applied it to SERVICE too.
2. **SERVICE/FIRD bounds are materially wider in reality.** WD-1 FIRD min = 0.28 vs published 0.44; max = 1.72 vs 1.53. This is a math-level explanation for why our SERVICE intervals miss in volatile months — they are calibrated against the wrong product type.
3. **FIRD WD-1 captures 43% of all FIRD actuals** (58,988 of 135,969). For SERVICE, nearly half the closing activity happens on the last business day. No bound calibrated on a 3-day window can be tight there.

### The talking point

> *"The published min/max table is derived from PRODUCT-side date activity. When we apply it to SERVICE, we under-bound by ~30%. With the predecessor's own FY23–FY25 raw data I derived a SERVICE-specific table — WD-1 (0.28, 1.72), WD-2 (0.20, 1.81), WD-3 (0.18, 1.82) — that is materially wider. Whether to widen the published bounds or split the table by product type is a defensible methodology decision."*